# 수집 대장을 읽다 — 무엇을 언제 받았나

> `notebooks/01-데이터수집/01.수집대장을읽다.ipynb` · 2026-08-31 · 이동원

**이 노트북이 답하는 것**: *"우리가 받은 자료가 어디까지 있고, 안 받은 것은 왜 안 받았나."*

수집을 자동으로 돌리면 언젠가 반드시 묻게 됩니다 — "어제 배치 돌았나?", "이 종목은
왜 3월부터 없지?", "실패한 건가 원래 없는 건가?" 그 답이 `collect_log` 표에 있습니다.

이 노트북은 **읽기만 합니다.** DB 를 바꾸지 않습니다.

## 0. 준비 — DB 를 읽기 전용으로 연다

⚠️ **읽기 전용(`mode=ro`)으로 엽니다.** 그냥 열면 노트북이 쓰기 주체가 되고,
수집 배치가 도는 중이라면 잠금을 다툽니다. 보기만 할 것이므로 그럴 이유가 없습니다.

In [1]:
import sqlite3
from pathlib import Path

# 저장소 루트 기준 경로로 적는다. 절대 경로를 찍으면 사람마다 달라지고,
# 이 저장소는 PUBLIC 이라 남의 폴더 구조까지 커밋된다.
DB = Path("../../data/krx_cache.db")
print("DB 파일: data/krx_cache.db")
print(f"크기   : {DB.stat().st_size / 1024 / 1024:,.0f} MB")
print()
print("⚠️ 이 파일은 .gitignore 라 저장소에 없다 — KRX 이용약관 제11조 ② 가 제3자")
print("   제공을 금지한다. 다시 돌리려면 scripts/fetch_krx.py 로 먼저 채운다.")

con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
con.row_factory = sqlite3.Row


def 질의(sql, params=()):
    """결과를 (칸 이름, 행 목록) 으로 돌려준다. 표로 찍기 좋게."""
    cur = con.execute(sql, params)
    cols = [d[0] for d in cur.description]
    return cols, cur.fetchall()


def 표(sql, params=(), 폭=None):
    """질의 결과를 고정폭 표로 찍는다. pandas 없이도 읽히게."""
    cols, rows = 질의(sql, params)
    폭 = 폭 or [max(len(str(c)), *(len(str(r[i])) for r in rows)) if rows else len(str(c))
               for i, c in enumerate(cols)]
    print("  " + " │ ".join(str(c).ljust(w) for c, w in zip(cols, 폭, strict=True)))
    print("  " + "─┼─".join("─" * w for w in 폭))
    for r in rows:
        print("  " + " │ ".join(str(r[i]).ljust(w) for i, w in enumerate(폭)))
    print("\n  %d행" % len(rows))

DB 파일: data/krx_cache.db
크기   : 1,579 MB

⚠️ 이 파일은 .gitignore 라 저장소에 없다 — KRX 이용약관 제11조 ② 가 제3자
   제공을 금지한다. 다시 돌리려면 scripts/fetch_krx.py 로 먼저 채운다.


## 1. 이 DB 에 무슨 표가 있나

표가 8개입니다. 크게 셋으로 나뉩니다.

| 무리 | 표 | 무엇 |
|---|---|---|
| **자료** | `daily_price` · `index_price` | 실제 시세와 지수 |
| **대장** | `collect_log` · `fetch_log` · `index_fetch_log` | 무엇을 언제 받았나 |
| **살림** | `call_budget` · `raw_response` · `robots_cache` | 호출 예산 · 원문 · 크롤링 가드 |

In [2]:
표("""
SELECT name AS 표이름,
       (SELECT COUNT(*) FROM pragma_table_info(m.name)) AS 칸수
FROM sqlite_master m
WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
ORDER BY name
""")

  표이름             │ 칸수
  ────────────────┼───
  call_budget     │ 5 
  collect_log     │ 9 
  daily_price     │ 15
  fetch_log       │ 3 
  index_fetch_log │ 4 
  index_price     │ 12
  raw_response    │ 9 
  robots_cache    │ 5 

  8행


## 2. 자료는 어디부터 어디까지 있나

**중도 소멸 종목까지 들고 있는 것이 중요합니다.** 지금 상장된 종목만 남기면
*생존 편향*이 생깁니다 — 수업 자료(`learning/09-finance-ml/1.금융머신러닝.ipynb`)가
말하는 그 편향입니다. 망한 회사를 빼고 백테스트하면 성과가 실제보다 부풀어 오릅니다.

In [3]:
표("""
SELECT COUNT(*)                AS 행,
       COUNT(DISTINCT code)    AS 종목,
       COUNT(DISTINCT bas_dd)  AS 거래일,
       MIN(bas_dd)             AS 시작,
       MAX(bas_dd)             AS 끝
FROM daily_price
""")

  행       │ 종목   │ 거래일  │ 시작       │ 끝       
  ────────┼──────┼──────┼──────────┼─────────
  9209812 │ 3677 │ 4097 │ 20100104 │ 20260825

  1행


In [4]:
# 마지막 거래일에 없는 종목 = 그 사이에 사라진 종목
표("""
SELECT COUNT(*) AS 중도소멸종목
FROM (
  SELECT code FROM daily_price
  GROUP BY code
  HAVING MAX(bas_dd) < (SELECT MAX(bas_dd) FROM daily_price)
)
""")

  중도소멸종목
  ──────
  910   

  1행


## 3. 대장 — 안 받은 이유를 다섯으로 가른다

*"안 받았다"* 를 한 덩어리로 두면 두 방향으로 틀립니다.

- 전부 **재시도**로 보면 → 휴장일마다 영원히 같은 호출을 태웁니다 (하루 한도 10,000회)
- 전부 **완료**로 보면 → 진짜 실패가 조용히 묻힙니다

| 상태 | 뜻 | 다시 받나 |
|---|---|---|
| `ok` | 받았다 | 아니오 |
| `empty` | 받아 봤는데 0건 (휴장일) | 아니오 |
| `error` | 시도했는데 실패 | 예, 3회까지 |
| `quota_exhausted` | 오늘 한도를 다 썼다 | 예, 내일 |
| `out_of_range` | 출처가 그 기간을 안 준다 | 아니오 |

In [5]:
표("""
SELECT source              AS 출처,
       status              AS 상태,
       COUNT(*)            AS 대상수,
       SUM(rows)           AS 행합계,
       MAX(last_success_at) AS 마지막성공
FROM collect_log
GROUP BY source, status
ORDER BY source, status
""")

  출처        │ 상태    │ 대상수  │ 행합계     │ 마지막성공              
  ──────────┼───────┼──────┼─────────┼────────────────────
  krx_index │ empty │ 246  │ 0       │ 2026-08-26T10:34:58
  krx_index │ ok    │ 4097 │ 195864  │ 2026-08-26T10:34:59
  krx_stock │ empty │ 492  │ 0       │ 2026-08-26T11:37:33
  krx_stock │ ok    │ 8194 │ 9209812 │ 2026-08-26T11:37:48

  4행


## 4. 대상 이름은 `시장/날짜` 다

KRX 는 **시장마다 API 가 따로**입니다. 하루치를 받으려면 KOSPI 한 번, KOSDAQ 한 번 —
**날짜당 2콜**입니다.

그래서 대장의 한 줄이 한 콜에 대응하도록 `KOSPI/20260826` 형태로 적습니다.
이렇게 해야 하루 한도(10,000회)와 대장이 어긋나지 않고, *"코스피는 받았는데 코스닥은
실패"* 를 표현할 수 있습니다.

⚠️ **여기에 함정이 있었습니다.** 옛 대장(`fetch_log`)에는 시장 칸이 없어서, 그걸 옮긴
4,343줄이 `20260826` 처럼 **날짜만** 들고 있었습니다. 지수 쪽은 `KOSPI/20260826` 이라
한 표 안에 두 규칙이 사는 상태였습니다. 2026-08-31 에 `daily_price` 를 실제로 세어
시장별로 다시 깔았습니다.

In [6]:
표("""
SELECT source AS 출처, target AS 대상, status AS 상태, rows AS 행,
       last_success_at AS 마지막성공
FROM collect_log
WHERE target LIKE '%/20260825' OR target LIKE '%/20260826'
ORDER BY source, target
""")

  출처        │ 대상              │ 상태    │ 행    │ 마지막성공              
  ──────────┼─────────────────┼───────┼──────┼────────────────────
  krx_index │ KOSPI/20260825  │ ok    │ 51   │ 2026-08-26T10:17:58
  krx_index │ KOSPI/20260826  │ empty │ 0    │ 2026-08-26T10:34:58
  krx_stock │ KOSDAQ/20260825 │ ok    │ 1823 │ 2026-08-26T09:36:48
  krx_stock │ KOSDAQ/20260826 │ empty │ 0    │ 2026-08-26T11:32:15
  krx_stock │ KOSPI/20260825  │ ok    │ 944  │ 2026-08-26T09:36:48
  krx_stock │ KOSPI/20260826  │ empty │ 0    │ 2026-08-26T11:32:15

  6행


### 다시 깐 결과가 시세 표와 맞는가

대장이 *"KOSPI 를 3,774,002행 받았다"* 고 말하면 시세 표에 정말 그만큼 있어야 합니다.
**한 행이라도 어긋나면 대장이 거짓말을 하는 것**이고, 그러면 화면도 거짓말을 합니다.

In [7]:
for market in ("KOSPI", "KOSDAQ"):
    대장 = con.execute(
        "SELECT SUM(rows) FROM collect_log WHERE source='krx_stock' AND target LIKE ?",
        (market + "/%",),
    ).fetchone()[0]
    실제 = con.execute(
        "SELECT COUNT(*) FROM daily_price WHERE market=?", (market,)
    ).fetchone()[0]
    print("  %-7s 대장 %9d · 시세표 %9d  → %s"
          % (market, 대장, 실제, "일치" if 대장 == 실제 else "어긋남"))

남은옛것 = con.execute(
    "SELECT COUNT(*) FROM collect_log "
    "WHERE source='krx_stock' AND target NOT LIKE '%/%'"
).fetchone()[0]
print("\n  날짜 전용(옛 형식) 줄 %d개 → %s"
      % (남은옛것, "깨끗" if 남은옛것 == 0 else "아직 남아 있다"))

  KOSPI   대장   3774002 · 시세표   3774002  → 일치


  KOSDAQ  대장   5435810 · 시세표   5435810  → 일치

  날짜 전용(옛 형식) 줄 0개 → 깨끗


## 5. 마지막 성공 시각 — 화면이 읽는 값

수집 현황 화면의 *"마지막 성공"* 은 이 값을 그대로 보여 줍니다.
그래서 **여기가 안 갱신되면 화면이 조용히 거짓말합니다** — 에러도 경고도 없이.

실제로 그런 상태였습니다. `krx_store` 가 시세를 저장하면서 옛 표(`fetch_log`)에만 쓰고
이 대장에는 쓰지 않았습니다. 수집은 멀쩡히 돌고 화면만 옛 시각에 멈춰 있었습니다.

In [8]:
표("""
SELECT source AS 출처,
       MAX(last_success_at) AS 마지막성공,
       MAX(CASE WHEN status='ok' THEN target END) AS 가장최근받은대상
FROM collect_log
GROUP BY source
""")

  출처        │ 마지막성공               │ 가장최근받은대상      
  ──────────┼─────────────────────┼───────────────
  krx_index │ 2026-08-26T10:34:59 │ KOSPI/20260825
  krx_stock │ 2026-08-26T11:37:48 │ KOSPI/20260825

  2행


## 6. 자료가 있는 마지막 날과 대장이 말하는 마지막 날

둘이 다를 수 있습니다. **받아 봤는데 0건**이면 대장에는 날짜가 남지만 시세 표에는
행이 없습니다. 장 마감 전에 받으면 이렇게 됩니다 — 자료가 아직 안 올라온 것이지
실패가 아닙니다.

In [9]:
마지막자료 = con.execute("SELECT MAX(bas_dd) FROM daily_price").fetchone()[0]
마지막대장 = con.execute(
    "SELECT MAX(SUBSTR(target, INSTR(target,'/')+1)) FROM collect_log "
    "WHERE source='krx_stock'"
).fetchone()[0]
print("  시세가 있는 마지막 날 :", 마지막자료)
print("  대장이 아는 마지막 날 :", 마지막대장)
print()

표("""
SELECT target AS 대상, status AS 상태, rows AS 행
FROM collect_log
WHERE source='krx_stock'
ORDER BY SUBSTR(target, INSTR(target,'/')+1) DESC, target
LIMIT 6
""")

  시세가 있는 마지막 날 : 20260825
  대장이 아는 마지막 날 : 20260826

  대상              │ 상태    │ 행   
  ────────────────┼───────┼─────
  KOSDAQ/20260826 │ empty │ 0   
  KOSPI/20260826  │ empty │ 0   
  KOSDAQ/20260825 │ ok    │ 1823
  KOSPI/20260825  │ ok    │ 944 
  KOSDAQ/20260824 │ ok    │ 1823
  KOSPI/20260824  │ ok    │ 942 

  6행


## 7. 정리

| 알게 된 것 | 값 (2026-08-31) |
|---|---|
| 시세 | 9,209,812행 · 3,677종목 · 4,097거래일 |
| 중도 소멸 종목 | 910 — 생존 편향을 피하려고 지우지 않는다 |
| 대장 | `krx_stock` 8,686줄 · `krx_index` 4,343줄 |
| 대장 ↔ 시세표 | 시장별·합계 모두 일치 |

**다음에 볼 것**

- `02-품질·전처리/` — 거래정지·정리매매·신규상장을 어떻게 가르나
- 호출 예산(`call_budget`)이 하루 한도를 어떻게 지키나

⚠️ 이 노트북의 숫자는 **2026-08-31 기준**입니다. 수집이 더 돌면 움직입니다.

In [10]:
con.close()
print("연결을 닫았다.")

연결을 닫았다.
